# Z+Jet Purity / Stability Diagnostic

**Purpose:** Run the absolute minimal Z+jet selection (no corrections, no systematics,
no reweighting) and compute purity/stability directly from the response matrix.

If purity/stability are still bad here, the problem is in the **event selection or binning**,
not in the corrections stack. If they look fine, start adding components back one at a time.

In [ ]:
%load_ext autoreload
%autoreload 2

## Cell 1 — Imports

In [ ]:
import sys
import importlib
import time
from pathlib import Path

import numpy as np
import awkward as ak
import hist
import matplotlib.pyplot as plt
import dask

from coffea.nanoevents import NanoAODSchema

# Make sure the package is importable from the notebook directory
repo_root = Path.cwd()
if not (repo_root / "src" / "zjet_corrections").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str((repo_root / "src").resolve()))

import zjet_corrections.notebook_utils as nbutils
from zjet_corrections.zjet_minimal_processor import ZJetMinimalProcessor

make_runner            = nbutils.make_runner
ensure_client          = nbutils.ensure_client
upload_package_if_casa = nbutils.upload_package_if_casa
format_time            = nbutils.format_time

NanoAODSchema.warn_missing_crossrefs = False

dask.config.set({
    "distributed.logging.distributed": "error",
    "distributed.logging.bokeh": "error",
    "distributed.logging.tornado": "error",
})

print("Imports OK")

## Cell 2 — Configuration

Set the era, pT bin, and whether to run in test mode (1–2 files only).

In [ ]:
ERA           = "2018"           # "2016", "2016APV", "2017", "2018"
DATASET       = "pythia"         # "pythia" or "herwig"
PREPENDSTR    = "root://xcache/" # set to "" for local files
EXECUTOR_MODE = "futures"        # "futures", "dask-local", "dask-casa"
CASA          = False

# Jet pT bin to study
PT_LO = 290.0
PT_HI = 400.0

# Run on a single file / single chunk only (for quick checks)
TEST           = True
CHUNKSIZE      = 50_000
CHUNKSIZE_TEST = 200_000

## Cell 3 — Build fileset (same samples as run_analysis.ipynb)

In [ ]:
paths       = nbutils.get_analysis_paths(repo_root)
sample_path = nbutils.SamplePath(ERA)

txt_files = sample_path.pythia[0] if DATASET == "pythia" else sample_path.herwig[0]

fileset = nbutils.build_fileset_from_txts(
    txt_files,
    paths.samples_mc_dir,
    PREPENDSTR,
    split_ht=True,
    ht_bins=nbutils.HT_BINS,
)

print(f"Fileset keys ({len(fileset)}):")
for k, v in fileset.items():
    print(f"  {k}: {len(v)} file(s)")

## Cell 4 — Create executor client

In [ ]:
client = ensure_client(casa=CASA, test=TEST, useDefault=False, executor_mode=EXECUTOR_MODE)
upload_package_if_casa(client, casa=CASA)

## Cell 5 — Run the minimal processor

In [ ]:
# In test mode: trim to one HT-bin dataset, one chunk — mirrors run_once() behaviour
run_fileset = fileset
if TEST:
    first_key   = list(fileset.keys())[0]
    run_fileset = {first_key: [fileset[first_key][0]]}
    print("TEST mode — running over:", run_fileset)

run = make_runner(
    use_dask  = client is not None,
    client    = client,
    chunksize = CHUNKSIZE_TEST if TEST else CHUNKSIZE,
    maxchunks = 1 if TEST else None,
)

start = time.time()
out = run(
    run_fileset,
    ZJetMinimalProcessor(pt_lo=PT_LO, pt_hi=PT_HI),
    treename="Events",
)
print(f"Done in {format_time(time.time() - start)}.")
print("Output keys:", list(out.keys()))

## Cell 6 — Unpack output & quick checks

`processor.Runner` accumulates across all datasets and chunks into a single flat dict —
no need to loop over HT bins.

In [ ]:
response_total = out["response"]
reco_total     = out["reco_mass"]
gen_total      = out["gen_mass"]
n_events_total = out["n_events"]

print(f"Total events after all cuts: {n_events_total:,}")

# Raw response matrix values
R = response_total.values()   # shape: (n_reco_bins, n_gen_bins)
print(f"Response matrix shape: {R.shape}")
print(f"Total weighted entries in response: {R.sum():.1f}")
print(f"Diagonal sum: {np.diag(R).sum():.1f}")
print(f"Rough overall diagonal fraction: {np.diag(R).sum() / (R.sum() + 1e-10):.3f}")

In [ ]:
# 1D distributions — these tell you if the mass spectrum looks sane
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

reco_vals  = reco_total.values()
gen_vals   = gen_total.values()
bin_edges  = reco_total.axes["reco_mass"].edges
centers    = (bin_edges[:-1] + bin_edges[1:]) / 2

ax1.step(centers, reco_vals, where='mid', label='Reco jet mass')
ax1.set_xlabel("Jet Mass [GeV]")
ax1.set_ylabel("Weighted entries")
ax1.set_title(f"Reco jet mass  (pT {PT_LO:.0f}–{PT_HI:.0f} GeV)")
ax1.legend()

gen_edges  = gen_total.axes["gen_mass"].edges
gen_centers = (gen_edges[:-1] + gen_edges[1:]) / 2
ax2.step(gen_centers, gen_vals, where='mid', color='tab:orange', label='Gen jet mass (matched)')
ax2.set_xlabel("Jet Mass [GeV]")
ax2.set_ylabel("Weighted entries")
ax2.set_title(f"Gen jet mass  (pT {PT_LO:.0f}–{PT_HI:.0f} GeV)")
ax2.legend()

plt.tight_layout()
plt.savefig("mass_1d_minimal.png", dpi=150)
plt.show()

## Cell 7 — Purity and Stability

In [ ]:
# Purity:    P_i = R[i,i] / sum_j R[i,j]   (fraction of reco-bin-i events whose gen mass is also in bin i)
# Stability: S_j = R[j,j] / sum_i R[i,j]   (fraction of gen-bin-j events whose reco mass is also in bin j)

row_sum = R.sum(axis=1)   # sum over gen bins → denominator for purity
col_sum = R.sum(axis=0)   # sum over reco bins → denominator for stability

purity    = np.diag(R) / (row_sum + 1e-10)
stability = np.diag(R) / (col_sum + 1e-10)

mass_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.step(mass_centers, purity,    where='mid', color='tab:blue',  lw=2, label='Minimal cuts only')
ax1.axhline(0.5, ls=':', color='k', label='0.5 threshold')
ax1.set_xlabel("Jet Mass [GeV] (Ungroomed)")
ax1.set_ylabel("Purity")
ax1.set_title(f"Purity — pT {PT_LO:.0f}–{PT_HI:.0f} GeV, {ERA}")
ax1.set_ylim(0, 1)
ax1.legend()

ax2.step(mass_centers, stability, where='mid', color='tab:orange', lw=2, label='Minimal cuts only')
ax2.axhline(0.5, ls=':', color='k', label='0.5 threshold')
ax2.set_xlabel("Jet Mass [GeV] (Ungroomed)")
ax2.set_ylabel("Stability")
ax2.set_title(f"Stability — pT {PT_LO:.0f}–{PT_HI:.0f} GeV, {ERA}")
ax2.set_ylim(0, 1)
ax2.legend()

plt.tight_layout()
plt.savefig("purity_stability_minimal.png", dpi=150)
plt.show()

print("\nPer-bin purity:")
for c, p, s in zip(mass_centers, purity, stability):
    flag = "<< BELOW 0.5" if p < 0.5 or s < 0.5 else ""
    print(f"  mass={c:5.0f} GeV   purity={p:.3f}   stability={s:.3f}  {flag}")

## Cell 8 — Response matrix (row-normalised)

In [ ]:
R_norm = R / (row_sum[:, None] + 1e-10)   # normalise each reco row

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(
    R_norm,
    origin='lower', aspect='auto', cmap='Blues',
    vmin=0, vmax=1,
    extent=[bin_edges[0], bin_edges[-1], bin_edges[0], bin_edges[-1]],
)
plt.colorbar(im, ax=ax, label='Fraction (row-normalised)')
ax.set_xlabel("Gen Jet Mass [GeV]")
ax.set_ylabel("Reco Jet Mass [GeV]")
ax.set_title(f"Response Matrix — Minimal cuts only\npT {PT_LO:.0f}–{PT_HI:.0f} GeV, {ERA}")
plt.tight_layout()
plt.savefig("response_matrix_minimal.png", dpi=150)
plt.show()

## Cell 9 — Interpretation guide

| Outcome | Likely cause | Next step |
|---|---|---|
| **Same bad purity/stability** as full processor | Event selection or binning | Inspect gen–reco ΔR and pT-ratio distributions below |
| **Better purity/stability** | Something in corrections/systematics | Re-add JEC → JER → reweighting one at a time |
| **Even worse** | Gen–reco matching broken | Check `lead_jet.matched_gen` is not None for most events |

## Cell 10 — Additional sanity checks (run on a small event loop)

These require access to per-event arrays, so we re-run on one file directly.

In [ ]:
from coffea.nanoevents import NanoEventsFactory

# Grab the first available file for a quick per-event diagnostic
first_key  = next(iter(fileset))
first_file = fileset[first_key][0]
print(f"Loading: {first_file}")

events_test = NanoEventsFactory.from_root(
    {first_file: "Events"},
    schemaclass=NanoAODSchema,
    entry_stop=50_000,
).events()

# Reproduce the selection inline
mu = events_test.Muon
good_mu = mu[
    (mu.pt > 20.) & (np.abs(mu.eta) < 2.4) & mu.tightId & (mu.pfRelIso04_all < 0.15)
]
has_os = (ak.num(good_mu) >= 2) & (ak.sum(good_mu.charge, axis=1) == 0)
ev = events_test[has_os]
gm = good_mu[has_os]
mu1 = gm[:, 0]; mu2 = gm[:, 1]
Z   = mu1 + mu2
z_ok = (Z.mass > 71.) & (Z.mass < 111.)
ev = ev[z_ok]; Z = Z[z_ok]; mu1 = mu1[z_ok]; mu2 = mu2[z_ok]

jets = ev.Jet
gj = jets[
    (jets.pt > 30.) & (np.abs(jets.eta) < 2.4) & (jets.jetId >= 2)
]
gj = gj[
    ak.all(gj.metric_table(mu1) > 0.4, axis=2)
    & ak.all(gj.metric_table(mu2) > 0.4, axis=2)
]
has_jet = ak.num(gj) >= 1
ev = ev[has_jet]; Z = Z[has_jet]; gj = gj[has_jet]
lead = gj[:, 0]
dphi_ok = np.abs(lead.delta_phi(Z)) > 2.7
ev = ev[dphi_ok]; lead = lead[dphi_ok]
pt_ok = (lead.pt > PT_LO) & (lead.pt < PT_HI)
ev = ev[pt_ok]; lead = lead[pt_ok]

matched_gen = lead.matched_gen
has_match   = ~ak.is_none(matched_gen)

n_total   = len(lead)
n_matched = int(ak.sum(has_match))
print(f"\nEvents after all cuts: {n_total}")
print(f"Matched fraction:      {n_matched/max(n_total,1):.3f}  ({n_matched}/{n_total})")
print("\nIf matched fraction < ~0.85, your matching is the problem.")

In [ ]:
if n_matched > 0:
    matched_reco_j = lead[has_match]
    matched_gen_j  = matched_gen[has_match]

    pt_ratio = ak.to_numpy(matched_reco_j.pt / matched_gen_j.pt)
    dR_vals  = ak.to_numpy(matched_reco_j.delta_r(matched_gen_j))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.hist(pt_ratio, bins=50, range=(0.5, 1.5), color='steelblue')
    ax1.axvline(1.0, ls='--', color='k')
    ax1.set_xlabel("pT_reco / pT_gen")
    ax1.set_ylabel("Events")
    ax1.set_title("Should be centred near 1.0")

    ax2.hist(dR_vals, bins=50, range=(0, 0.5), color='darkorange')
    ax2.axvline(0.2, ls='--', color='k', label='0.2')
    ax2.set_xlabel("ΔR(reco, gen)")
    ax2.set_ylabel("Events")
    ax2.set_title("Should be << 0.4 for well-matched jets")
    ax2.legend()

    plt.tight_layout()
    plt.savefig("matching_sanity.png", dpi=150)
    plt.show()
else:
    print("No matched events — check NanoAOD cross-references and sample size.")